# 06 · Causal (Masked) Self-Attention

**Belgedeki bölüm:** Causal (Masked) Self Attention (ve Self Attention › Q, K, V)

- Her token öğrenilen üç matrisle üç role bürünür: **Query** (ne arıyorum?), **Key** (ne sunuyorum?), **Value** (hangi bilgiyi veririm?).
- `softmax(Q·Kᵀ / √dₖ) · V`
- Causal maske: diyagonalin üstü (gelecek) −∞ yapılır, softmax sonrası 0 olur → model kopya çekemez (next-token prediction).

In [ ]:
import torch
from usta_embedding import UstaEmbedding
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)

# UstaEmbedding = nn.Embedding + RoPE pozisyon bilgisi (04_Positional_Encoding, usta_embedding.py)
torch.manual_seed(1)
u_embedding = UstaEmbedding(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_embedding(tokens)
sentence_meanings

![image.png](https://lena-voita.github.io/resources/lectures/seq2seq/transformer/qkv_attention_formula-min.png)

In [ ]:
q_weights = torch.nn.Linear(4, 3, bias=False)
k_weights = torch.nn.Linear(4, 3, bias=False)
v_weights = torch.nn.Linear(4, 3, bias=False)

q_of_sentence = q_weights(sentence_meanings)
k_of_sentence = k_weights(sentence_meanings)
v_of_sentence = v_weights(sentence_meanings)
print(q_weights.weight)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape

In [ ]:
k_of_sentence.shape

Self-Attention

Artık her kelimenin sadece cümle içinde değil bulunduğu bağlamda anlamları

In [ ]:
#Cümledeki her kelimenin, diğer tüm kelimelerle olan ham "benzerlik" veya "alaka" puanını hesaplar.
attention_scores = q_of_sentence @ k_of_sentence.T #Benzerlik Matrisi (attention_scores)

#Bölme işlemi, dot product sonuçlarının çok büyümesini ve gradyanların patlamasını (Softmax'in doyuma ulaşmasını) engeller.
#Softmax: Bu ham puanları $0$ ile $1$ arasına çeker ve her satırın toplamını 1 yapar.
attention_weights = torch.softmax(attention_scores / k_of_sentence.shape[-1] ** 0.5, dim=1)#Ölçekleme ve Olasılığa Dönüştürme

#Hesapladığın o hassas ağırlıkları, gerçek bilgi/anlamı taşıyan Value (V) matrisi ile çarpıyorsun.
#Bu adımda her kelime, cümledeki diğer kelimelerden ne kadar bilgi alacağına karar verir.
#context_vector, her kelimenin orijinal embedding'inden çok daha akıllı, 
#çevresindeki kelimelerin anlamını da içine çekmiş "bağlamsal" (context-aware) halidir.
context_vector = attention_weights @ v_of_sentence #Bilginin Harmanlanması
context_vector

In [ ]:
from plot_tokens import plot_tokens

u_sentences = [
  {
    "words": q_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "blue",
  },
  {
    "words": k_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "purple",
  },
  {
    "words": v_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "orange",
  },
  {
    "words": context_vector.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "green",
  },
]

plot_tokens(u_sentences, "Query, Key, Value and Context Vector Space")

### Hazır modül: UstaSelfAttention (maskesiz)
Yukarıdaki Q, K, V → softmax → ·V adımları `usta_self_attention.py` içinde tek katman.

In [ ]:
from usta_self_attention import UstaSelfAttention

torch.manual_seed(1)
self_attention = UstaSelfAttention(embedding_dim=4, output_dim=3)
self_attention(sentence_meanings)

## Causal Self Attention

In [ ]:
attention_weights

![softmax](https://i.ytimg.com/vi/EuZZ6plg2Tk/maxresdefault.jpg)

In [ ]:
mask = torch.tril(torch.ones(7, 7))
mask

In [ ]:
attention_weights = torch.randn(7, 7)

masked_attention_weights = attention_weights.masked_fill(mask == 0, -torch.inf)
masked_attention_weights

In [ ]:
softmaxed_attention_weights = torch.softmax(masked_attention_weights, dim=1)
softmaxed_attention_weights

### Hazır modül: UstaCausalAttention
Yukarıdaki adımların hepsi (Q, K, V → maske → softmax → ·V) `usta_causal_attention.py` içinde tek bir katman.

In [ ]:
from usta_causal_attention import UstaCausalAttention

torch.manual_seed(1)
causal_attention = UstaCausalAttention(embedding_dim=4, output_dim=4, context_length=32)

context_vector = causal_attention(sentence_meanings)
context_vector